[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedorShind/EMRG/blob/main/docs/tutorials/qaoa_maxcut_mitigation.ipynb)

# QAOA with EMRG

This notebook builds a small MaxCut QAOA circuit. It has more non-Clifford structure than the VQE example, so EMRG has a different recipe decision to make.

The problem is choosing the recipe, not calling Mitiq. EMRG is the boring layer in between: inspect, choose, render.

## Install

Colab needs the preview extra for the optional simulator cell. In a local checkout, use the project virtual environment instead.

In [ ]:
!pip install -q "emrg[preview]"

## Imports

In [ ]:
from math import isinf

from qiskit import QuantumCircuit

from emrg import analyze_circuit, generate_recipe

## Build a MaxCut circuit

The graph is fixed, and the angles are fixed. That keeps the recipe decision stable every time the notebook runs.

In [ ]:
graph_edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
gamma = 0.63
beta = 0.29

qc = QuantumCircuit(5, 5)
qc.h(range(5))

for _ in range(2):
    for i, j in graph_edges:
        qc.rzz(2 * gamma, i, j)
    for qubit in range(5):
        qc.rx(2 * beta, qubit)

qc.measure(range(5), range(5))
qc.draw("text")

## Analyze the circuit

Same first step as before: inspect the circuit and turn it into features the policy can use.

In [ ]:
features = analyze_circuit(qc)

feature_summary = {
    "qubits": features.num_qubits,
    "depth": features.depth,
    "total gates": features.total_gate_count,
    "multi-qubit gates": features.multi_qubit_gate_count,
    "noise category": features.noise_category,
    "non-Clifford gates": features.non_clifford_count,
    "non-Clifford fraction": f"{features.non_clifford_fraction:.1%}",
    "PEC overhead estimate": f"{features.pec_overhead_estimate:.2f}x",
}

for name, value in feature_summary.items():
    print(f"{name:24} {value}")

## Ask EMRG for a recipe

The default policy considers CDR for circuits above 30% non-Clifford gates and depth 12 through 40. This QAOA circuit fits that window, so EMRG recommends CDR instead of the ZNE recipe used in the VQE example.

In [ ]:
def print_recipe(recipe):
    print(f"Technique:          {recipe.technique}")
    print(f"Factory:            {recipe.factory_name or 'none'}")
    print(f"Scale factors:      {recipe.scale_factors or 'none'}")
    print(f"Scaling method:     {recipe.scaling_method or 'none'}")
    print(f"Estimated overhead: {recipe.estimated_overhead:.1f}x")
    if recipe.factory_kwargs:
        print(f"Factory kwargs:     {dict(recipe.factory_kwargs)}")
    if recipe.warnings:
        print("Warnings:")
        for warning in recipe.warnings:
            print(f"  - {warning}")
    else:
        print("Warnings:           none")


result = generate_recipe(qc)
print_recipe(result.recipe)

## Look at the generated code

The output is still Mitiq code. For CDR, the generated code includes the training-circuit setup and a backend executor hook you provide.

In [ ]:
def show_code_excerpt(code, max_lines=45):
    lines = code.splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")


show_code_excerpt(result.code)

## Optional preview

Preview is simulation, not hardware evidence. The current CDR preview path can be skipped by simulator conversion for this RZZ circuit, so the cell handles that as a normal outcome.

In [ ]:
preview_result = generate_recipe(qc, preview=True, noise_level=0.01)
p = preview_result.preview

if p is None:
    print("Preview skipped: preview dependencies are not installed")
elif p.ideal_value is None:
    print(f"Preview skipped: {p.warning}")
else:
    reduction = "infinite" if isinf(p.error_reduction) else f"{p.error_reduction:.1f}x"
    print(f"Technique:  {p.technique}")
    print(f"Ideal:      {p.ideal_value:+.4f}")
    print(f"Noisy:      {p.noisy_value:+.4f}  (error: {p.noisy_error:.4f})")
    print(f"Mitigated:  {p.mitigated_value:+.4f}  (error: {p.mitigated_error:.4f})")
    print(f"Reduction:  {reduction}")
    if p.warning:
        print(f"Note:       {p.warning}")

## Policy note

This notebook uses the built-in default policy. For a real workflow, keep policy choices in a JSON file and pass it to EMRG instead of editing the package code.

In [ ]:
# Example only. This assumes you created emrg-policy.json with:
# emrg policy init emrg-policy.json
#
# result = generate_recipe(qc, policy="emrg-policy.json")

## Summary

The API did not change between the examples. The circuit did.

The VQE-style ansatz was short, so EMRG rendered a ZNE recipe. This QAOA circuit has enough non-Clifford structure at the right depth for CDR, so EMRG renders a CDR recipe with a small training set.